# evtools — Tutorial

**Evidence Theory Tools** — a Python library for working with belief functions in the Dempster-Shafer / Transferable Belief Model framework.

This tutorial uses a running example: a sensor classifying aerial targets as **airplane** (a), **helicopter** (h), or **rocket** (r).

**Sections**
1. Building a BBA
2. Accessing values
3. Conversions (bel, pl, b, q)
4. Conjunctive and disjunctive weights (v, w)
5. Simple MFs
6. Combination rules
7. Decombination
8. Correction mechanisms
9. Display formats
10. Low-level conversions API

**References**
- P. Smets. *The application of the matrix calculus to belief functions*, IJAR, 31(1–2):1–30, 2002.
- T. Denœux. *Conjunctive and disjunctive combination of belief functions induced by nondistinct bodies of evidence*, AI, 172:234–264, 2008.
- D. Mercier, B. Quost, T. Denœux. *Refined modeling of sensor reliability in the belief function framework using contextual discounting*, Information Fusion, 9(2):246–258, 2008.
- F. Pichon, D. Mercier, É. Lefèvre, F. Delmotte. *Proposition and learning of some belief function contextual correction mechanisms*, IJAR, 72:4–42, 2016.

## Setup

In [ ]:
import numpy as np
from evtools.dsvector import DSVector, Kind
from evtools.combinations import crc, dempster, drc, cautious, bold, decombine_crc, decombine_drc
from evtools.corrections import (
    discount, contextual_discount, theta_contextual_discount,
    contextual_reinforce, contextual_dediscount, contextual_dereinforce,
    contextual_negate,
)
from evtools.display import repr_plain, repr_html, repr_latex

frame = ["a", "h", "r"]  # airplane, helicopter, rocket
print("evtools loaded successfully")

---
## 1. Building a Basic Belief Assignment (BBA)

A BBA is a function $m: 2^\Omega \to [0,1]$ with $\sum_{A \subseteq \Omega} m(A) = 1$.

**Three constructors are available:**

In [ ]:
# from_focal: human-friendly string keys.
# Missing mass is automatically assigned to Ω = {a, h, r}.
m = DSVector.from_focal(frame, {"a": 0.5, "r": 0.5})
m  # Jupyter renders the HTML table automatically via _repr_html_

In [ ]:
# from_dense: numpy array in binary index order (Smets 2002).
# Index i → subset whose members are the atoms at the bit positions set in i.
# For frame=[a,h,r]: 0=∅, 1={a}, 2={h}, 3={a,h}, 4={r}, 5={a,r}, 6={h,r}, 7={a,h,r}
array = np.array([0.0, 0.5, 0.0, 0.0, 0.5, 0.0, 0.0, 0.0])
m2 = DSVector.from_dense(frame, array)
m2

In [ ]:
# from_sparse: dict of frozensets.
m3 = DSVector.from_sparse(frame, {
    frozenset({"a"}): 0.5,
    frozenset({"r"}): 0.5,
})
m3

In [ ]:
# Subnormal BBA: m(∅) > 0 (allowed in the TBM, represents internal conflict)
m_sub = DSVector.from_focal(frame, {"": 0.1, "a": 0.3, "r": 0.4, "a,h,r": 0.2}, complete=False)
print(f"is_valid: {m_sub.is_valid}")
m_sub

---
## 2. Accessing Values

In [ ]:
print("sparse dict :", m.sparse)
print("dense array :", m.dense)
print("m({a})      :", m[frozenset({"a"})])
print("m({h})      :", m[frozenset({"h"})], " ← not a focal element")
print("n_focal     :", m.n_focal)
print("is_valid    :", m.is_valid)

In [ ]:
print("Iterating over focal elements:")
for subset, value in m:
    label = "{"+", ".join(sorted(subset))+"}" if subset else "∅"
    print(f"  m({label}) = {value:.4f}")

---
## 3. Conversions

All standard representations are available via `.to(Kind)` or shortcuts.
The column header in the display adapts to the kind (`m`, `bel`, `pl`, `b`, `q`, `v`, `w`).

In [ ]:
m.to_bel()  # Belief function

In [ ]:
m.to_pl()  # Plausibility function

In [ ]:
m.to_b()  # Commonality function

In [ ]:
m.to_q()  # Implicability function

In [ ]:
# Round-trip consistency check
for kind in [Kind.BEL, Kind.PL, Kind.B, Kind.Q]:
    back = m.to(kind).to(Kind.M)
    ok = np.allclose(back.dense, m.dense, atol=1e-10)
    print(f"  m → {kind.value:>3} → m : {'✓ OK' if ok else '✗ MISMATCH'}")

---
## 4. Conjunctive and Disjunctive Weights (v, w)

The conjunctive weight function $w$ and disjunctive weight function $v$ require
a **subnormal BBA** ($m(\emptyset) > 0$), so that $b(A) > 0$ for all $A \subseteq \Omega$.

They are defined via a Möbius transform on $\ln q$ (for $w$) and $\ln b$ (for $v$),
and are essential for the **Cautious** and **Bold** combination rules (Denoeux 2008).

In [ ]:
# Subnormal BBA required: m(∅) > 0
m_sub2 = DSVector.from_focal(frame, {"": 0.1, "a": 0.3, "r": 0.4, "a,h,r": 0.2}, complete=False)
print(f"is_valid: {m_sub2.is_valid}")
m_sub2

In [ ]:
# Disjunctive weight function v
m_sub2.to_v()

In [ ]:
# Conjunctive weight function w
m_sub2.to_w()

In [ ]:
# Round-trip consistency for v and w
for kind in [Kind.V, Kind.W]:
    back = m_sub2.to(kind).to(Kind.M)
    ok = np.allclose(back.dense, m_sub2.dense, atol=1e-10)
    print(f"  m_sub → {kind.value} → m : {'✓ OK' if ok else '✗ MISMATCH'}")

---
## 5. Simple MFs

Simple MFs are the elementary building blocks of correction mechanisms:
- $A^\beta$ (`DSVector.simple`): focal sets $\Omega$ (mass $\beta$) and $A$ (mass $1-\beta$) — used in CR, CdR, CN
- $A_\beta$ (`DSVector.negative_simple`): focal sets $\emptyset$ (mass $\beta$) and $A$ (mass $1-\beta$) — used in CD, CdD

In [ ]:
# Simple MF A^β (positive)
DSVector.simple(frame, frozenset({"a"}), beta=0.6)

In [ ]:
# Negative simple MF A_β
DSVector.negative_simple(frame, frozenset({"a"}), beta=0.4)

---
## 6. Combination Rules

| | All sources reliable | At least one reliable |
|---|---|---|
| **Distinct sources** | `crc` (`&`) / `dempster` (`@`) | `drc` (`\|`) |
| **Nondistinct sources** | `cautious` | `bold` |

In [ ]:
s1 = DSVector.from_focal(frame, {"a": 0.5, "r": 0.5})
s2 = DSVector.from_focal(frame, {"h": 0.3, "r": 0.4, "a,h,r": 0.3})

# CRC — distinct, both reliable (operator &)
m12_crc = s1 & s2
print(f"Conflict m(∅) = {m12_crc[frozenset()]:.4f}")
m12_crc

In [ ]:
# Dempster's rule — normalized CRC (operator @)
s1 @ s2

In [ ]:
# DRC — at least one reliable (operator |)
s1 | s2

In [ ]:
# Cautious — nondistinct, both reliable; commutative, associative, idempotent
s1_nd = DSVector.from_focal(frame, {"a": 0.3, "h": 0.2, "a,h,r": 0.5})
s2_nd = DSVector.from_focal(frame, {"a": 0.4, "r": 0.1, "a,h,r": 0.5})
m_caut = cautious(s1_nd, s2_nd)
print(f"Idempotent: cautious(m,m)==m → {np.allclose(cautious(s1_nd,s1_nd).dense, s1_nd.dense, atol=1e-10)}")
m_caut

In [ ]:
# Bold — nondistinct, at least one reliable (requires subnormal BBAs)
b1 = DSVector.from_focal(frame, {"": 0.1, "a": 0.4, "a,h,r": 0.5}, complete=False)
b2 = DSVector.from_focal(frame, {"": 0.2, "r": 0.3, "a,h,r": 0.5}, complete=False)
bold(b1, b2)

---
## 7. Decombination

Inverse operations to remove a previously combined BBA.
The result may not be a valid BBA — always check `.is_valid`.

In [ ]:
# decombine_crc: m1 6∩ m2
m1 = DSVector.from_focal(frame, {"a": 0.4, "a,h,r": 0.6})
m2 = DSVector.from_focal(frame, {"h": 0.3, "a,h,r": 0.7})
m12 = crc(m1, m2)
m1_recovered = decombine_crc(m12, m2)

print(f"is_valid  : {m1_recovered.is_valid}")
print(f"Recovers m1: {np.allclose(m1_recovered.dense, m1.dense, atol=1e-6)}")
m1_recovered

---
## 8. Correction Mechanisms

Corrections adjust a BBA based on knowledge about the source quality (reliability, truthfulness).

| Mechanism | Notation | Source assumption |
|-----------|---------|-------------------|
| `discount(m, β)` | $m_S \cup \Omega_{\beta}$ | single reliability degree |
| `contextual_discount(m, β)` (CD) | $m_S \cup \bigcup_{A} A_{\beta_A}$ | reliability per singleton |
| `theta_contextual_discount(m, β)` | general Θ partition | reliability per coarsening |
| `contextual_reinforce(m, β)` (CR) | $m_S \cap \bigcap_{A} A^{\beta_A}$ | contextual positive liar |
| `contextual_dediscount(m, β)` (CdD) | inverse of CD | undo a prior CD |
| `contextual_dereinforce(m, β)` (CdR) | inverse of CR | undo a prior CR |
| `contextual_negate(m, β)` (CN) | equivalence rule | contextual non-truthful |

In [ ]:
m = DSVector.from_focal(frame, {"a": 0.5, "r": 0.5})

# Classical discounting: β=0.6, source 60% reliable
discount(m, beta=0.6)

In [ ]:
# Contextual discounting: sensor unreliable only when target is airplane
# β_a=0.6, β_h=1.0, β_r=1.0  (Example 1, Case 1 of Mercier et al. 2008)
betas_cd = {frozenset({"a"}): 0.6, frozenset({"h"}): 1.0, frozenset({"r"}): 1.0}
mcd = contextual_discount(m, betas_cd)
# Interpretation: mass on {r} partially transferred to {a,r}
# because if the true target is airplane, sensor may declare it a rocket
mcd

In [ ]:
# Θ-contextual discounting: coarser partition Θ = {{a}, {h,r}}
theta_contextual_discount(m, {frozenset({"a"}): 0.4, frozenset({"h","r"}): 0.9})

In [ ]:
# Contextual Reinforcement (CR) — dual of CD, uses CRC
contextual_reinforce(m, betas_cd)

In [ ]:
# CdD: inverse of CD — check .is_valid after decombination
mdd = contextual_dediscount(mcd, betas_cd)
print(f"is_valid      : {mdd.is_valid}")
print(f"CdD reverts CD: {np.allclose(mdd.dense, m.dense, atol=1e-6)}")
mdd

In [ ]:
# Contextual Negating (CN): source non-truthful with probability 1−β
contextual_negate(m, {frozenset({"a"}): 0.7})

In [ ]:
# Pure negation: β=0, A=∅  →  m(B) becomes m(B̄) for all B
contextual_negate(m, {frozenset(): 0.0})

---
## 9. Display Formats

Four output formats are available. In Jupyter, `DSVector` renders as an HTML table automatically (via `_repr_html_`). Use `.display(fmt)` to request a specific format.

In [ ]:
m = DSVector.from_focal(frame, {"a": 0.5, "r": 0.5})

# HTML (default in Jupyter)
m

In [ ]:
# Plain text — no colors, for logging and files
print(repr_plain(m))

In [ ]:
# Column header adapts to the kind
print("--- Belief function ---")
print(repr_plain(m.to_bel()))
print("\n--- Plausibility function ---")
print(repr_plain(m.to_pl()))
print("\n--- Implicability function ---")
print(repr_plain(m.to_q()))

In [ ]:
# LaTeX — ready to paste in a paper
print(repr_latex(m))

In [ ]:
# Explicit format selection
print(m.display("plain"))

---
## 10. Low-level Conversions API

All conversions are available as standalone functions on numpy arrays,
using the Fast Möbius Transform (Smets 2002, Section 3).

In [ ]:
from evtools.conversions import mtob, mtopl, mtoq, mtobel

m_array = np.array([0.0, 0.5, 0.0, 0.0, 0.5, 0.0, 0.0, 0.0])
print("m     :", m_array)
print("bel   :", mtobel(m_array))
print("pl    :", mtopl(m_array))
print("b     :", mtob(m_array))
print("q     :", mtoq(m_array))

---
## Summary

| Module | Key objects / functions |
|--------|------------------------|
| `evtools.dsvector` | `DSVector`, `Kind`, `.simple()`, `.negative_simple()`, `.is_valid`, `.display(fmt)` |
| `evtools.combinations` | `crc` (`&`), `dempster` (`@`), `drc` (`\|`), `cautious`, `bold`, `decombine_crc`, `decombine_drc` |
| `evtools.corrections` | `discount`, `contextual_discount`, `theta_contextual_discount`, `contextual_reinforce`, `contextual_dediscount`, `contextual_dereinforce`, `contextual_negate` |
| `evtools.display` | `repr_ansi`, `repr_plain`, `repr_html`, `repr_latex` |
| `evtools.conversions` | `mtob`, `mtopl`, `mtoq`, `btom`, ... (42 functions) |
| `evtools.constants` | `ZERO_MASS`, `MASS_TOL`, `VALID_TOL`, `DISPLAY_TOL` |